# EmpowerLens — S08: does synthetic data help? (Kaggle GPU)

Trains the **D12 model** (MentalRoBERTa, T14 settings: lr 3e-5, 5 epochs, weight decay 0.1, max length 512,
batch 16) two ways, **3 seeds each** (42, 1337, 2024):

| Condition | Training rows |
|---|---|
| `train_only` | `data/splits/train.csv` (2,024) |
| `train_synth` | `data/splits_synth/train.csv` = the same 2,024 + 232 rows of `synthetic_train_en_final.jsonl` (S10) |

Val is the same 253 rows in both. For every condition × seed:
1. binary model, D12 failed-run rule (val macro-F1 < 0.5 → re-run once), then the **D13 threshold**
   (`src.fit_binary_threshold --target-fpr 0.15`, fitted on val);
2. multilabel model (per-type thresholds fitted on val during training);
3. `src.evaluate --splits data/splits` on both (val + test; test is read only there);
4. `src.eval_supplementary` on the 376 S03 supplementary-evaluation rows;
5. the checkpoints are deleted (only JSON results are kept; no weights leave Kaggle).

**Before running:** Accelerator **GPU (T4 or P100)**, **Internet: On**, Kaggle secret **`HF_TOKEN`**
(`mental/mental-roberta-base` is gated). Use **Save Version → Save & Run All**: 12 trainings take about
4–5 hours on a T4.

**Resumable:** a condition × seed whose `supp_eval_*.json` already exists in `/kaggle/working/s08/` is skipped.

**After it finishes:** download `s08.zip` and put the `s08/` folder in `D3/`.

In [ ]:
# 1. HF token, single GPU, clone the repo into /kaggle/temp, install.
import os
from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded.")
except Exception as e:
    print(f"[warn] no HF_TOKEN secret ({e}) - mental-roberta-base will fail to download.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # avoid multi-GPU load hangs on T4 x2

REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "lumia-space"
!rm -rf /kaggle/temp/empowerlens && mkdir -p /kaggle/temp && git clone --depth 1 --branch $BRANCH $REPO_URL /kaggle/temp/empowerlens
%cd /kaggle/temp/empowerlens
!pip install -q -r requirements-transformer.txt
!git log --oneline -1 && ls data/splits_synth src/eval_supplementary.py && python -c "import torch; print('cuda:', torch.cuda.is_available())"

In [ ]:
# 2. Config + helpers (re-run this cell first after any kernel restart).
import os, json, shutil, subprocess, time
from pathlib import Path
os.chdir("/kaggle/temp/empowerlens")

MODEL  = "mental/mental-roberta-base"
TAG    = MODEL.split("/")[-1]
SEEDS  = [42, 1337, 2024]
CONDS  = {"train_only": "data/splits", "train_synth": "data/splits_synth"}
OUT    = Path("/kaggle/working/s08")
CK     = Path("/kaggle/temp/ckpt_s08")
RECIPE = f"--model {MODEL} --lr 3e-5 --epochs 5 --weight-decay 0.1 --max-length 512 --batch-size 16"
OUT.mkdir(parents=True, exist_ok=True)

def sh(cmd):
    print(f"\n$ {cmd}", flush=True)
    t = time.time()
    subprocess.run(cmd, shell=True, check=True)
    print(f"[done in {(time.time() - t) / 60:.1f} min]", flush=True)

def meta(ck):
    return json.loads((Path(ck) / "meta.json").read_text())

print(json.loads(Path("data/splits_synth/split_manifest.json").read_text())["n_train_synthetic"], "synthetic rows in train_synth")

In [ ]:
# 3. Smoke test (~3 min): tiny binary run on the synthetic splits. Writes nothing to OUT.
sh(f"python -m src.train_transformer --task binary --model {MODEL} --seed 42 --smoke --max-length 128 "
   f"--splits data/splits_synth --out /kaggle/temp/smoke")
print("SMOKE OK")
shutil.rmtree("/kaggle/temp/smoke", ignore_errors=True)

In [ ]:
# 4. Main loop: condition x seed. Resumable (skips finished units).
for cond, splits in CONDS.items():
    for seed in SEEDS:
        out = OUT / cond
        done = out / "supp" / f"supp_eval_{cond}_{seed}.json"
        if done.exists():
            print(f"[skip] {cond} seed {seed}: already done")
            continue
        print(f"\n===== {cond} | seed {seed} =====", flush=True)
        ck = CK / cond
        b_ck, m_ck = ck / f"binary_{TAG}_{seed}", ck / f"multilabel_{TAG}_{seed}"
        recipe = f"{RECIPE} --seed {seed} --splits {splits}"

        if not ((b_ck / "meta.json").exists() and "binary_threshold" in meta(b_ck)):
            reruns = 0
            sh(f"python -m src.train_transformer --task binary {recipe} --out {ck}")
            f1 = meta(b_ck)["val_metrics"]["eval_macro_f1"]
            if f1 < 0.5:   # D12 failed-run rule: re-run once with identical settings
                print(f"[failed-run rule] val macro-F1 {f1:.3f} < 0.5 -> re-running once")
                reruns = 1
                sh(f"python -m src.train_transformer --task binary {recipe} --out {ck}")
            sh(f"python -m src.fit_binary_threshold --checkpoint {b_ck} --target-fpr 0.15")
            m = meta(b_ck); m["failed_run_reruns"] = reruns
            (b_ck / "meta.json").write_text(json.dumps(m, indent=2))
        if not (m_ck / "meta.json").exists():
            sh(f"python -m src.train_transformer --task multilabel {recipe} --out {ck}")

        for c in (b_ck, m_ck):   # test is read only by src.evaluate, always on the frozen benchmark splits
            sh(f"python -m src.evaluate --checkpoint {c} --splits data/splits --out {out / 'eval'}")
        sh(f"python -m src.eval_supplementary --binary {b_ck} --multilabel {m_ck} --condition {cond} "
           f"--out {out / 'supp'}")
        (out / "meta").mkdir(parents=True, exist_ok=True)
        for c in (b_ck, m_ck):
            shutil.copy(c / "meta.json", out / "meta" / f"{c.name}.json")
        shutil.rmtree(b_ck, ignore_errors=True); shutil.rmtree(m_ck, ignore_errors=True)
print("\nALL UNITS DONE")

In [ ]:
# 5. Summary tables + zip for download (JSON results only, no weights).
sh(f"python -m src.synth_effect --raw {OUT}")
for f in ("summary.csv", "paired.csv", "supp_per_class.csv", "per_seed.csv"):
    shutil.copy(Path("results/results_synth_effect") / f, OUT / f)
!cd /kaggle/working && rm -f s08.zip && zip -qr s08.zip s08 && ls -la s08 && du -sh s08.zip